# 00 Shared ML table

The one table both ML people use, so all models see exactly the same data.

- **One row per (batch, sample time > 0).**
- **Features** (`X`): day, day-0 values, and for each feed the amount added so far (`cum_*`) and in the last 24 h (`last24h_*`). All are known for test batches too. No measurement after day 0 is used.
- **Labels**: the 4 measured values (`vcd`, `tcd`, `substrate`, `byproduct`). Empty for test batches.
- **Split**: each row gets its batch's `role` (`dev` / `holdout` / `test`) and `fold` (0-4 for dev) from `outputs/splits.csv`. A batch is never divided.

Measured volume is left out on purpose: it is almost exactly `volume0` + the feeds, so it would only duplicate them.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]
FEEDS = ["feed_substrate", "feed_b", "feed_c", "feed_d"]

## Build the table
For each batch: the first sample is the day-0 starting state. Every later sample time becomes one row.

In [ ]:
def build_table(df):
    rows = []
    for b, g in df.groupby("batch_id"):
        g = g.sort_values("elapsed_time")
        samples = g[g.volume.notna()]                      # sample rows (test ones have volume only)
        d0 = samples.iloc[0]                               # day-0 starting state
        feeds = g[g[FEEDS].notna().any(axis=1)]            # feed events
        for _, s in samples.iloc[1:].iterrows():           # every later sample time
            t = s.elapsed_time
            before = feeds[feeds.elapsed_time < t]         # feeds that happened before this sample
            last_day = before[before.elapsed_time >= t - 1]
            r = dict(batch_id=b, elapsed_time=t, day=t, vcd0=d0.vcd, viability0=d0.vcd / d0.tcd,
                     substrate0=d0.substrate, byproduct0=d0.byproduct, volume0=d0.volume)
            for f in FEEDS:
                r[f"cum_{f}"] = before[f].sum()            # feed added so far
                r[f"last24h_{f}"] = last_day[f].sum()      # feed added in the last 24 h
            for c in TARGETS:
                r[c] = s[c]                                # label (NaN for test batches)
            rows.append(r)
    return pd.DataFrame(rows)


df = pd.read_csv(ROOT / "data" / "bioreaction_remaining.csv")
tab = build_table(df)
FEATURES = [c for c in tab.columns if c not in TARGETS + ["batch_id", "elapsed_time"]]

## Attach the frozen split

In [ ]:
splits = pd.read_csv(ROOT / "outputs" / "splits.csv")
tab = tab.merge(splits[["batch_id", "role", "fold"]], on="batch_id", how="left")

dev = tab[tab.role == "dev"].reset_index(drop=True)          # train + 5-fold CV
holdout = tab[tab.role == "holdout"].reset_index(drop=True)  # closed until the end
test = tab[tab.role == "test"].reset_index(drop=True)        # to predict (no labels)

## Checks

In [ ]:
print("features:", FEATURES)
print(tab.groupby("role").agg(rows=("batch_id", "size"), batches=("batch_id", "nunique")))
print("dev rows per fold:", dev.groupby("fold").batch_id.nunique().to_dict(), "(batches)")

assert tab.role.notna().all(), "a batch is missing from splits.csv"
assert not tab[FEATURES].isna().any().any(), "NaN in features"
assert dev[TARGETS].notna().all().all() and holdout[TARGETS].notna().all().all(), "labels missing"
assert test[TARGETS].isna().all().all(), "test batches should have no labels"
assert tab.groupby("batch_id").role.nunique().max() == 1, "a batch is split across roles"
print("all checks passed")